In [1]:
import sqlite3

# Simple SQL demo using SQLite (no Spark).
# Everything runs in-memory and resets if you restart the kernel.
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row


def run(sql: str, params: tuple | None = None):
    """Execute SQL and pretty-print results for SELECT queries."""
    sql = sql.strip().rstrip(";")
    cur = conn.cursor()
    cur.execute(sql, params or ())

    if cur.description is None:
        conn.commit()
        print("OK")
        return None

    rows = cur.fetchall()
    print(f"{len(rows)} row(s)")
    for r in rows[:20]:
        print(dict(r))
    if len(rows) > 20:
        print("... (showing first 20)")
    return rows


print("SQLite ready")

SQLite ready


In [3]:
# Create tables
run("""
CREATE TABLE customers (
  customer_id   INTEGER PRIMARY KEY,
  customer_name TEXT NOT NULL,
  city          TEXT NOT NULL
)
""")

run("""
CREATE TABLE products (
  product_id   INTEGER PRIMARY KEY,
  product_name TEXT NOT NULL,
  category     TEXT NOT NULL,
  price        REAL NOT NULL
)
""")

run("""
CREATE TABLE orders (
  order_id    INTEGER PRIMARY KEY,
  customer_id INTEGER NOT NULL,
  product_id  INTEGER NOT NULL,
  quantity    INTEGER NOT NULL,
  order_date  TEXT NOT NULL,
  FOREIGN KEY(customer_id) REFERENCES customers(customer_id),
  FOREIGN KEY(product_id)  REFERENCES products(product_id)
)
""")

run("SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name")

OK
OK
OK
3 row(s)
{'name': 'customers', 'sql': 'CREATE TABLE customers (\n  customer_id   INTEGER PRIMARY KEY,\n  customer_name TEXT NOT NULL,\n  city          TEXT NOT NULL\n)'}
{'name': 'orders', 'sql': 'CREATE TABLE orders (\n  order_id    INTEGER PRIMARY KEY,\n  customer_id INTEGER NOT NULL,\n  product_id  INTEGER NOT NULL,\n  quantity    INTEGER NOT NULL,\n  order_date  TEXT NOT NULL,\n  FOREIGN KEY(customer_id) REFERENCES customers(customer_id),\n  FOREIGN KEY(product_id)  REFERENCES products(product_id)\n)'}
{'name': 'products', 'sql': 'CREATE TABLE products (\n  product_id   INTEGER PRIMARY KEY,\n  product_name TEXT NOT NULL,\n  category     TEXT NOT NULL,\n  price        REAL NOT NULL\n)'}


In [4]:
# Insert sample data
run("DELETE FROM orders")
run("DELETE FROM customers")
run("DELETE FROM products")

run("""
INSERT INTO customers(customer_id, customer_name, city) VALUES
  (1, 'Asha',    'Bengaluru'),
  (2, 'Rahul',   'Mumbai'),
  (3, 'Neha',    'Delhi'),
  (4, 'Ibrahim', 'Mumbai')
""")

run("""
INSERT INTO products(product_id, product_name, category, price) VALUES
  (101, 'Laptop',       'Electronics', 85000.00),
  (102, 'Headphones',   'Electronics',  2500.00),
  (103, 'Office Chair', 'Home',        12000.00),
  (104, 'T-Shirt',      'Clothing',      799.00)
""")

run("""
INSERT INTO orders(order_id, customer_id, product_id, quantity, order_date) VALUES
  (1001, 1, 101, 1, '2024-01-05'),
  (1002, 2, 102, 2, '2024-01-05'),
  (1003, 2, 104, 3, '2024-01-06'),
  (1004, 3, 103, 1, '2024-01-10'),
  (1005, 4, 102, 1, '2024-01-11'),
  (1006, 1, 104, 2, '2024-01-12')
""")

run("SELECT * FROM customers ORDER BY customer_id")
run("SELECT * FROM products ORDER BY product_id")
run("SELECT * FROM orders ORDER BY order_id")

OK
OK
OK
OK
OK
OK
4 row(s)
{'customer_id': 1, 'customer_name': 'Asha', 'city': 'Bengaluru'}
{'customer_id': 2, 'customer_name': 'Rahul', 'city': 'Mumbai'}
{'customer_id': 3, 'customer_name': 'Neha', 'city': 'Delhi'}
{'customer_id': 4, 'customer_name': 'Ibrahim', 'city': 'Mumbai'}
4 row(s)
{'product_id': 101, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 85000.0}
{'product_id': 102, 'product_name': 'Headphones', 'category': 'Electronics', 'price': 2500.0}
{'product_id': 103, 'product_name': 'Office Chair', 'category': 'Home', 'price': 12000.0}
{'product_id': 104, 'product_name': 'T-Shirt', 'category': 'Clothing', 'price': 799.0}
6 row(s)
{'order_id': 1001, 'customer_id': 1, 'product_id': 101, 'quantity': 1, 'order_date': '2024-01-05'}
{'order_id': 1002, 'customer_id': 2, 'product_id': 102, 'quantity': 2, 'order_date': '2024-01-05'}
{'order_id': 1003, 'customer_id': 2, 'product_id': 104, 'quantity': 3, 'order_date': '2024-01-06'}
{'order_id': 1004, 'customer_id': 3, 'prod

In [5]:
# Simple SELECT + WHERE + ORDER BY
run("""
SELECT order_id, customer_id, product_id, quantity, order_date
FROM orders
WHERE order_date >= '2024-01-06'
ORDER BY order_date, order_id
""")

4 row(s)
{'order_id': 1003, 'customer_id': 2, 'product_id': 104, 'quantity': 3, 'order_date': '2024-01-06'}
{'order_id': 1004, 'customer_id': 3, 'product_id': 103, 'quantity': 1, 'order_date': '2024-01-10'}
{'order_id': 1005, 'customer_id': 4, 'product_id': 102, 'quantity': 1, 'order_date': '2024-01-11'}
{'order_id': 1006, 'customer_id': 1, 'product_id': 104, 'quantity': 2, 'order_date': '2024-01-12'}


In [7]:
# JOIN + computed column (line amount)
run("""
SELECT
  o.order_id,
  c.customer_name,
  c.city,
  p.product_name,
  p.category,
  o.quantity,
  p.price,
  (o.quantity * p.price) AS line_amount,
  o.order_date
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id
ORDER BY o.order_date, o.order_id
""")

6 row(s)
{'order_id': 1001, 'customer_name': 'Asha', 'city': 'Bengaluru', 'product_name': 'Laptop', 'category': 'Electronics', 'quantity': 1, 'price': 85000.0, 'line_amount': 85000.0, 'order_date': '2024-01-05'}
{'order_id': 1002, 'customer_name': 'Rahul', 'city': 'Mumbai', 'product_name': 'Headphones', 'category': 'Electronics', 'quantity': 2, 'price': 2500.0, 'line_amount': 5000.0, 'order_date': '2024-01-05'}
{'order_id': 1003, 'customer_name': 'Rahul', 'city': 'Mumbai', 'product_name': 'T-Shirt', 'category': 'Clothing', 'quantity': 3, 'price': 799.0, 'line_amount': 2397.0, 'order_date': '2024-01-06'}
{'order_id': 1004, 'customer_name': 'Neha', 'city': 'Delhi', 'product_name': 'Office Chair', 'category': 'Home', 'quantity': 1, 'price': 12000.0, 'line_amount': 12000.0, 'order_date': '2024-01-10'}
{'order_id': 1005, 'customer_name': 'Ibrahim', 'city': 'Mumbai', 'product_name': 'Headphones', 'category': 'Electronics', 'quantity': 1, 'price': 2500.0, 'line_amount': 2500.0, 'order_date': 

In [8]:
# GROUP BY + aggregates (sales by category)
run("""
SELECT
  p.category,
  COUNT(DISTINCT o.order_id) AS orders,
  SUM(o.quantity)            AS units,
  ROUND(SUM(o.quantity * p.price), 2) AS revenue
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC
""")

# HAVING example: only categories with >= 2 units
run("""
SELECT
  p.category,
  SUM(o.quantity) AS units
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
HAVING SUM(o.quantity) >= 2
ORDER BY units DESC
""")

3 row(s)
{'category': 'Electronics', 'orders': 3, 'units': 4, 'revenue': 92500.0}
{'category': 'Home', 'orders': 1, 'units': 1, 'revenue': 12000.0}
{'category': 'Clothing', 'orders': 2, 'units': 5, 'revenue': 3995.0}
2 row(s)
{'category': 'Clothing', 'units': 5}
{'category': 'Electronics', 'units': 4}


[<sqlite3.Row at 0x108912440>, <sqlite3.Row at 0x108913490>]

In [9]:
# GROUP BY + join: top customers by revenue
run("""
SELECT
  c.customer_id,
  c.customer_name,
  c.city,
  ROUND(SUM(o.quantity * p.price), 2) AS revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id
GROUP BY c.customer_id, c.customer_name, c.city
ORDER BY revenue DESC
""")

4 row(s)
{'customer_id': 1, 'customer_name': 'Asha', 'city': 'Bengaluru', 'revenue': 86598.0}
{'customer_id': 3, 'customer_name': 'Neha', 'city': 'Delhi', 'revenue': 12000.0}
{'customer_id': 2, 'customer_name': 'Rahul', 'city': 'Mumbai', 'revenue': 7397.0}
{'customer_id': 4, 'customer_name': 'Ibrahim', 'city': 'Mumbai', 'revenue': 2500.0}


In [12]:
# Window function demo: rank products by revenue within category
run("""
WITH product_sales AS (
  SELECT
    p.category,
    p.product_id,
    p.product_name,
    SUM(o.quantity) AS units,
    SUM(o.quantity * p.price) AS revenue
  FROM orders o
  JOIN products p ON o.product_id = p.product_id
  GROUP BY p.category, p.product_id, p.product_name
)
SELECT
  category,
  product_id,
  product_name,
  units,
  ROUND(revenue, 2) AS revenue,
  DENSE_RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS revenue_rank_in_category
FROM product_sales
ORDER BY category, revenue_rank_in_category, product_id
""")

4 row(s)
{'category': 'Clothing', 'product_id': 104, 'product_name': 'T-Shirt', 'units': 5, 'revenue': 3995.0, 'revenue_rank_in_category': 1}
{'category': 'Electronics', 'product_id': 101, 'product_name': 'Laptop', 'units': 1, 'revenue': 85000.0, 'revenue_rank_in_category': 1}
{'category': 'Electronics', 'product_id': 102, 'product_name': 'Headphones', 'units': 3, 'revenue': 7500.0, 'revenue_rank_in_category': 2}
{'category': 'Home', 'product_id': 103, 'product_name': 'Office Chair', 'units': 1, 'revenue': 12000.0, 'revenue_rank_in_category': 1}


In [ ]:
# LEFT JOIN demo: customers with/without orders
run("""
SELECT
  c.customer_id,
  c.customer_name,
  COUNT(o.order_id) AS orders
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY orders DESC, c.customer_id
""")

# Example of an index (helps joins/filtering as data grows)
run("CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON orders(customer_id)")
run("CREATE INDEX IF NOT EXISTS idx_orders_product_id  ON orders(product_id)")

4 row(s)
{'customer_id': 1, 'customer_name': 'Asha', 'orders': 2}
{'customer_id': 2, 'customer_name': 'Rahul', 'orders': 2}
{'customer_id': 3, 'customer_name': 'Neha', 'orders': 1}
{'customer_id': 4, 'customer_name': 'Ibrahim', 'orders': 1}
OK
OK


26/04/07 14:05:09 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

In [ ]:
# Cleanup (optional)
# run("DROP TABLE IF EXISTS orders")
# run("DROP TABLE IF EXISTS products")
# run("DROP TABLE IF EXISTS customers")
pass